# Financial Transaction Pipeline — Exploratory Data Analysis

This notebook covers:
1. Data overview & quality checks
2. Monthly spend trends (line chart)
3. Category breakdown (pie + bar)
4. Channel distribution
5. Anomaly detection — Z-score vs IQR comparison
6. User-level segmentation
7. Heatmap: category × month spend

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

from pipeline.extract import extract
from pipeline.transform import transform

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 1. Generate & Transform Data (in-memory — no DB required)

In [ ]:
raw = extract(num_transactions=20_000)
df  = transform(raw)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df['amount'].describe().round(2)

## 2. Monthly Spend Trends

In [ ]:
completed = df[df['status'] == 'completed'].copy()
monthly = (
    completed.groupby(['year', 'month'])['amount']
    .agg(['sum', 'count', 'mean'])
    .reset_index()
    .rename(columns={'sum': 'total_spend', 'count': 'txn_count', 'mean': 'avg_spend'})
)
monthly['period'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)
monthly = monthly.sort_values('period')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax1.plot(monthly['period'], monthly['total_spend'], marker='o', linewidth=2, color='steelblue')
ax1.fill_between(monthly['period'], monthly['total_spend'], alpha=0.15, color='steelblue')
ax1.set_title('Monthly Total Spend', fontsize=14, fontweight='bold')
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax1.set_ylabel('Total Spend (USD)')
ax1.tick_params(axis='x', rotation=45)

ax2.bar(monthly['period'], monthly['txn_count'], color='coral', alpha=0.8)
ax2.set_title('Monthly Transaction Count', fontsize=14, fontweight='bold')
ax2.set_ylabel('# Transactions')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('monthly_trends.png', bbox_inches='tight')
plt.show()

## 3. Category Breakdown

In [ ]:
cat_spend = (
    completed.groupby('merchant_category')['amount']
    .sum()
    .sort_values(ascending=False)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Pie
ax1.pie(
    cat_spend,
    labels=cat_spend.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
)
ax1.set_title('Spend Share by Category', fontsize=14, fontweight='bold')

# Bar
bars = ax2.barh(cat_spend.index[::-1], cat_spend.values[::-1], color=sns.color_palette('muted', len(cat_spend)))
ax2.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax2.set_title('Total Spend per Category', fontsize=14, fontweight='bold')
ax2.set_xlabel('Total Spend (USD)')

plt.tight_layout()
plt.savefig('category_breakdown.png', bbox_inches='tight')
plt.show()

## 4. Channel Distribution

In [ ]:
channel_data = completed.groupby('channel').agg(
    txn_count=('amount', 'count'),
    total_spend=('amount', 'sum'),
    avg_spend=('amount', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, label in zip(axes, ['txn_count', 'total_spend', 'avg_spend'],
                           ['Transaction Count', 'Total Spend ($)', 'Avg Spend ($)']):
    bars = ax.bar(channel_data['channel'], channel_data[col], color=sns.color_palette('pastel'))
    ax.set_title(label, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Transaction Channel Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('channel_analysis.png', bbox_inches='tight')
plt.show()

## 5. Anomaly Detection — Z-Score vs IQR

In [ ]:
from analytics.anomalies import zscore_anomalies, iqr_anomalies

z_flagged  = zscore_anomalies(df=completed, threshold=3.0)
iq_flagged = iqr_anomalies(df=completed, multiplier=1.5)

print(f'Z-score anomalies : {len(z_flagged):,}')
print(f'IQR anomalies     : {len(iq_flagged):,}')
print(f'Union             : {len(set(z_flagged.transaction_id) | set(iq_flagged.transaction_id)):,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cat in zip(axes, completed['merchant_category'].unique()[:2]):
    cat_df = completed[completed['merchant_category'] == cat]
    z_cat  = z_flagged[z_flagged['merchant_category'] == cat]
    iq_cat = iq_flagged[iq_flagged['merchant_category'] == cat]

    ax.scatter(cat_df.index, cat_df['amount'], s=10, alpha=0.4, color='steelblue', label='Normal')
    ax.scatter(z_cat.index, z_cat['amount'], s=60, color='red', label='Z-score anomaly', zorder=5)
    ax.scatter(iq_cat.index, iq_cat['amount'], s=60, marker='^', color='orange', label='IQR anomaly', zorder=5)
    ax.set_title(f'{cat} — Anomaly Detection', fontweight='bold')
    ax.set_xlabel('Transaction Index')
    ax.set_ylabel('Amount ($)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('anomaly_detection.png', bbox_inches='tight')
plt.show()

## 6. Category × Month Spend Heatmap

In [ ]:
pivot = completed.pivot_table(
    index='merchant_category',
    columns='month',
    values='amount',
    aggfunc='sum',
    fill_value=0
)

plt.figure(figsize=(14, 7))
sns.heatmap(
    pivot,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Total Spend (USD)'},
)
plt.title('Category × Month Spend Heatmap', fontsize=15, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Merchant Category')
plt.tight_layout()
plt.savefig('category_month_heatmap.png', bbox_inches='tight')
plt.show()

## 7. User-Level Spend Segmentation

In [ ]:
user_stats = (
    completed.groupby('user_id')['amount']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'lifetime_spend', 'count': 'txn_count', 'mean': 'avg_spend'})
)
user_stats['segment'] = pd.qcut(
    user_stats['lifetime_spend'],
    q=4,
    labels=['Bronze', 'Silver', 'Gold', 'Platinum']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

user_stats['segment'].value_counts().plot.bar(ax=axes[0], color=sns.color_palette('Set2'), rot=0)
axes[0].set_title('User Segments by Spend Quartile', fontweight='bold')
axes[0].set_ylabel('# Users')

axes[1].scatter(
    user_stats['txn_count'],
    user_stats['lifetime_spend'],
    c=user_stats['segment'].cat.codes,
    cmap='Set2',
    alpha=0.6,
    s=30
)
axes[1].set_xlabel('Transaction Count')
axes[1].set_ylabel('Lifetime Spend ($)')
axes[1].set_title('Spend vs Transaction Frequency', fontweight='bold')

plt.tight_layout()
plt.savefig('user_segmentation.png', bbox_inches='tight')
plt.show()

display(user_stats.groupby('segment')[['lifetime_spend','txn_count','avg_spend']].mean().round(2))